In [1]:
cd ..

/home/serafeim/Desktop/bet


In [18]:

from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.last_activity_generator import generate_last_activity
from prepare_data_inference import prepare_data_inference
from pyspark.ml.functions import vector_to_array
import mlflow 
import numpy as np


In [28]:
def compute_metrics(df, threshold):
    pred = df.withColumn(
        "pred_label",
        (F.col("p_churn") >= threshold).cast("int")
    )
    metrics = pred.groupBy("next_7d_churn_idx", "pred_label").count()
    tp = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 1").select("count").first()
    fp = metrics.filter("next_7d_churn_idx = 0 AND pred_label = 1").select("count").first()
    fn = metrics.filter("next_7d_churn_idx = 1 AND pred_label = 0").select("count").first()

    tp = tp[0] if tp else 0
    fp = fp[0] if fp else 0
    fn = fn[0] if fn else 0

    precision = float(np.round(tp / (tp + fp) ,2)) if (tp + fp) > 0 else 0.0
    recall = float(np.round(tp / (tp + fn), 2)) if (tp + fn) > 0 else 0.0
    f1 = (float(np.round( 2 * precision * recall / (precision + recall) , 2))
        if (precision + recall) > 0 else 0.0
    )

    return precision, recall, f1

In [4]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()

player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/11 13:04:17 WARN Utils: Your hostname, serafeim-virtual-machine, resolves to a loopback address: 127.0.1.1; using 10.100.2.34 instead (on interface ens192)
26/03/11 13:04:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 13:04:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')

In [6]:
test_date = '2024-05-20'
data_inference = prepare_data_inference(test_date)
data_inference.show(3)

failed_withdrawals_30d
deposit_count_30d
withdrawal_count_30d
withdrawal_ratio
+----------+--------------------+---------------------+------------------+------------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|player_idx|net_amount_result_7d|net_amount_result_30d|   balance_30d_ago|    balance_7d_ago|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|
+----------+--------------------+---------------------+------------------+------------------+----------------------+-----------------+--------------------+----------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+
|       964|              243.12|              1813

In [7]:
def compare(df1,df2): 
   assert (( df1.exceptAll(df2).count() == 0) & (df2.exceptAll(df1).count() == 0))

m1 = player_behavior.filter(F.col('reference_date')==test_date).select('player_idx').join(data_inference, how='inner', on='player_idx', )
compare(m1, data_inference)

In [8]:
mlflow.set_experiment("batch-mode experiment")

loaded_model = mlflow.spark.load_model(
    "models:/SparkLogisticRegression/latest"
)

from mlflow.tracking import MlflowClient

client = MlflowClient()

model_version = client.get_latest_versions(
    name="SparkLogisticRegression",
    #stages=["Production"]
)[0]

run_id = model_version.run_id
run = mlflow.get_run(run_id)

print(run_id)


2026/03/11 13:06:32 INFO mlflow.spark: URI 'models:/SparkLogisticRegression/latest/sparkml' does not point to the current DFS.
2026/03/11 13:06:32 INFO mlflow.spark: File 'models:/SparkLogisticRegression/latest/sparkml' not found on DFS. Will attempt to upload the file.


01a375f69d32429c9972841ec5ef87ab


/tmp/ipykernel_325863/980503102.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  model_version = client.get_latest_versions(


In [9]:

test_start = run.data.params["test_start"]
test_end = run.data.params["test_end"]
test_df =  (player_behavior.filter(F.col('reference_date')>=test_start)
        .join(player_snapshot, on="player_idx", how="left")
        .join(labels, on=["player_idx", "reference_date"], how="inner")
         .withColumn("next_7d_churn_idx", F.col("next_7d_churn").cast("int")))

print(test_start, test_end)

data_inference_ml = (player_behavior.select('player_idx','reference_date').filter(F.col('reference_date')==test_date)
.join(data_inference, how ='inner', on='player_idx')
.join(player_snapshot, on="player_idx", how="inner")
)

with mlflow.start_run(run_name=test_date):
    test_preds = (loaded_model.transform(data_inference_ml)
    .withColumn("p_churn", vector_to_array("probability")[1]))

    



2024-06-01 2024-06-22


In [10]:
def result_display(preds):
    preds = preds.select('player_idx','reference_date', 'p_churn', 'prediction')
    preds = preds.withColumn('risk_level', 
        F.when(F.col('p_churn')>=0.8, 'High')
        .when(F.col('p_churn')>=0.6, 'Medium')
        .when(F.col('p_churn')>=0.4, 'Low')
        .otherwise(F.lit('None')))
    return preds

In [11]:
results = result_display(test_preds)

In [12]:
results.show(4)

26/03/11 13:07:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+--------------+-------------------+----------+----------+
|player_idx|reference_date|            p_churn|prediction|risk_level|
+----------+--------------+-------------------+----------+----------+
|       233|    2024-05-20| 0.0782302578183034|       0.0|      None|
|       684|    2024-05-20|0.05043746605713162|       0.0|      None|
|       729|    2024-05-20| 0.2500839267519488|       0.0|      None|
|       745|    2024-05-20| 0.3773493401415742|       0.0|      None|
+----------+--------------+-------------------+----------+----------+
only showing top 4 rows


In [13]:
dates = (
    test_df
    .select("reference_date")
    .distinct()
    .orderBy("reference_date")
    .collect()
)

dates = [r.reference_date for r in dates]

In [40]:
results = []
threshold = 0.8

for d in dates:

    daily_df = test_df.filter(F.col("reference_date") == d)

    preds = (
         loaded_model.transform(daily_df)
        .withColumn("p_churn", vector_to_array("probability")[1])
    )
    preds = preds.withColumn('risk_level', 
        F.when(F.col('p_churn')>=0.8, 'High')
        .when(F.col('p_churn')>=0.6, 'Medium')
        .when(F.col('p_churn')>=0.4, 'Low')
        .otherwise(F.lit('None')))

    precision, recall, f1 = compute_metrics(preds, threshold)

    results.append({
        "date": str(d),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "num_players": preds.count(),
        "num_flagged": preds.filter(F.col("prediction") == 1).count(),
        "num_churned_": preds.filter(F.col("next_7d_churn_idx") == 1).count(),
        
        "num_churned_high_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'High')).count(),

        "num_churned_med_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'Medium')).count(),

        "num_churned_low_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'Low')).count() ,

        "num_churned_no_risk": (preds.filter(F.col("next_7d_churn_idx") == 1)
        .filter(F.col("risk_level") == 'None')).count() ,

   
    })

In [41]:
backtest_df = spark.createDataFrame(results)
backtest_df.orderBy("date").show(5)

+----------+----+------------+---------------------+--------------------+--------------------+-------------------+-----------+-----------+---------+------+
|      date|  f1|num_churned_|num_churned_high_risk|num_churned_low_risk|num_churned_med_risk|num_churned_no_risk|num_flagged|num_players|precision|recall|
+----------+----+------------+---------------------+--------------------+--------------------+-------------------+-----------+-----------+---------+------+
|2024-06-01|0.72|          56|                   36|                   6|                   3|                 11|         44|        651|     0.82|  0.64|
|2024-06-02| 0.7|          64|                   39|                   6|                   5|                 14|         47|        653|     0.83|  0.61|
|2024-06-03|0.72|          67|                   41|                   7|                   4|                 15|         47|        654|     0.87|  0.61|
|2024-06-04|0.68|          72|                   41|            

## Probability Calibration (model reliability)

In [ ]:
preds = preds.withColumn(
    "prob_bin",
    F.floor(F.col("p_churn") * 10) / 10
)

calibration = (
    preds
    .groupBy("prob_bin")
    .agg(
        F.avg("next_7d_churn_idx").alias("actual_rate"),
        F.count("*").alias("players")
    )
    .orderBy("prob_bin")
)
calibration.show()

## 2. Data Drift Monitoring

Your model assumes that feature distributions stay stable.

In [ ]:
. What Is Still Missing

You should add player counts per risk group.

Example:

risk	players	churners	churn_rate
high	44	36	0.82